# Analyse unifiée des entraînements

Analyse des artefacts existants sans charger les poids. Les tâches, politiques de labels et protocoles incompatibles sont séparés. Les résultats test servent à décrire les performances ; sélectionner les modèles sur la validation.

## 1 — Configuration
Les filtres `None` acceptent tous les runs. Modifier cette cellule puis exécuter de haut en bas. Les analyses détaillées sont limitées par `MAX_DETAIL_RUNS`.

La racine par défaut est `results/radio_noise`. `CLASSIFICATION_FILTER` distingue RR de RH ; `MODEL_FILTER`, `FOLD_FILTER` et `NOISE_FILTER` permettent de naviguer dans les expériences. Les comparaisons globales incluent tous les folds disponibles, indépendamment de la limite des diagnostics individuels.

In [ ]:
from pathlib import Path
import sys
import os
# Fonctionne depuis le projet, le package ou le dossier notebooks.
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "src/clusterprep").is_dir() or (p / "clusterprep/src/clusterprep").is_dir())
PACKAGE = PROJECT if (PROJECT / "src/clusterprep").is_dir() else PROJECT / "clusterprep"
sys.path.insert(0, str(PACKAGE / "src"))
ROOT_DIR = Path(os.environ.get("CLUSTERPREP_ANALYSIS_ROOT", str(PACKAGE.parent / "results/radio_noise")))
RECURSIVE = True
ENABLE_WIDGETS = True  # nécessite ipywidgets ; filtres de configuration sinon
TASK_FILTER = None  # binary / multilabel
CLASSIFICATION_FILTER = None  # rr_binary / rh_binary / de_nde / rr_rh
MODEL_FILTER = None  # fusion / dual_encoder_cnn
FOLD_FILTER = None  # entier ou liste, ex. [0, 2, 4]
NOISE_FILTER = None  # gaussian_pixel / beam_correlated
EXPERIMENT_FILTER = None
MODALITY_FILTER = TRANSFER_FILTER = CONFIGURATION_FILTER = None
SPLIT = "test"  # validation pour sélectionner des modèles
COHORT = "all"  # all, paired, raw_only
METRICS = ["macro_f1", "roc_auc_macro", "average_precision_macro", "per_label.RR.f1", "per_label.RH.f1", "accuracy"]
MAX_DETAIL_RUNS = 100  # nombre maximum de runs détaillés à afficher
MAX_COMPARISON_CONFIGS = 100  # conserver tous les folds des configurations affichées
DETAIL_RUN_NAMES = None  # noms de dossiers sans suffixe horodatage/identifiant
DETAIL_RUN_IDS = None  # sélection technique facultative, conservée pour compatibilité
SHOW_GALLERIES = False
MAX_GALLERY_CLUSTERS = 8
CHANDRA_ONLY = False
GALLERY_VIEW = "overlay"  # overlay, radio, chandra
PROCESSED_ROOT = PACKAGE.parent / "processed"
RADIO_TEMPLATE = "{cluster_id}/{cluster_id}_RAW_processed_size128_v2.fits"
CHANDRA_TEMPLATE = "{cluster_id}/{cluster_id}_CHANDRA_processed_size128_v2.fits"
LEGACY_ADVANCED = False  # audits candidats, bootstrap et contribution CHANDRA
EXPORT_FIGURES = True
EXPORT_TABLES = True
EXPORT_ROOT = PACKAGE.parent / "analysis_exports"


## 2 — Découverte et sélection
Les erreurs restent visibles. Un statut absent est `unverified`, jamais assimilé à une exécution terminée.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from clusterprep import analysis as a

all_runs, issues = a.discover_runs(ROOT_DIR, recursive=RECURSIVE)
runs = a.filter_runs(all_runs, task_type=TASK_FILTER, task=CLASSIFICATION_FILTER,
    noise_method=NOISE_FILTER, model=MODEL_FILTER,
    experiment_id=EXPERIMENT_FILTER, fold=FOLD_FILTER, modalities=MODALITY_FILTER,
    transfer_strategy=TRANSFER_FILTER, configuration_id=CONFIGURATION_FILTER)
registry = a.registry_table(runs)
figures = {}
print(f"{len(all_runs)} runs détectés, {len(runs)} sélectionnés")
if not registry.empty:
    display(registry.groupby("task_type").size().rename("runs"))
    print(f"{registry.experiment_id.nunique()} expériences, {registry.configuration_id.nunique()} configurations")
    display(registry.groupby(["task", "model", "noise_method"], dropna=False).agg(
        runs=("run_id", "size"), folds=("fold", "nunique")))
    display(a.display_table(registry.drop(columns=["config", "artifacts"])))
display(issues)
detail_runs = [r for r in runs if (DETAIL_RUN_IDS is None or r.run_id in DETAIL_RUN_IDS)
               and (DETAIL_RUN_NAMES is None or r.run_name in DETAIL_RUN_NAMES)][:MAX_DETAIL_RUNS]


### Appliquer les filtres interactifs (optionnels)

In [ ]:
read_filters = None
if ENABLE_WIDGETS:
    try:
        controls, read_filters = a.filter_controls(all_runs, defaults={
            "task_type": TASK_FILTER, "task": CLASSIFICATION_FILTER,
            "noise_method": NOISE_FILTER, "model": MODEL_FILTER, "experiment_id": EXPERIMENT_FILTER,
            "fold": FOLD_FILTER, "modalities": MODALITY_FILTER, "transfer_strategy": TRANSFER_FILTER,
            "configuration_id": CONFIGURATION_FILTER})
        display(controls)
    except ImportError:
        print("ipywidgets indisponible : les filtres de la cellule de configuration restent actifs.")


In [ ]:
# Appliquer : réexécuter cette cellule et les suivantes après modification des widgets.
if read_filters is not None:
    runs = a.filter_runs(all_runs, **read_filters())
    registry = a.registry_table(runs)
    display(a.display_table(registry))
detail_runs = [r for r in runs if (DETAIL_RUN_IDS is None or r.run_id in DETAIL_RUN_IDS)
               and (DETAIL_RUN_NAMES is None or r.run_name in DETAIL_RUN_NAMES)][:MAX_DETAIL_RUNS]

# Les vues globales incluent tous les folds ; la limite détaillée ne concerne que
# les diagnostics individuels, appariements et galeries.
comparison_ids = list(dict.fromkeys(r.configuration_id for r in runs))[:MAX_COMPARISON_CONFIGS]
comparison_runs = [r for r in runs if r.configuration_id in comparison_ids]
print(f"Comparaison : {len(comparison_ids)} configurations, {len(comparison_runs)} runs")


## 3 — Vue globale
Les agrégations excluent les runs non terminés. Moyenne puis écart-type échantillonnal entre folds ; les répétitions d’un même fold sont moyennées avant agrégation. Un seul fold donne un écart-type indéfini.

In [ ]:
metrics = a.metrics_table(runs, SPLIT, COHORT)
summary = a.aggregate_metrics(metrics, METRICS)
display(a.display_table(metrics))
display(a.display_table(summary))
comparison_metrics = a.metrics_table(comparison_runs, SPLIT, COHORT)
if comparison_runs:
    figures["model_comparison/summary"] = a.plot_performance(comparison_metrics, METRICS[0])
    plt.show()


## 4 — Historiques individuels et agrégés
Les phases restent distinctes. La moyenne utilise seulement les epochs présentes, sans prolongement des courbes ; `count` indique le nombre de runs contribuant. Les lignes grises marquent les meilleures epochs, les orange les transitions.

In [ ]:
for run in detail_runs:
    print(run.path)

In [ ]:
epochs = a.epoch_table(runs)
display(a.display_table(a.aggregate_history(epochs)))
if detail_runs:
    figures["training_curves/individual"] = a.plot_history(detail_runs)
    figures["training_curves/folds"] = a.plot_history(comparison_runs, aggregate=True)
    plt.show()


## 5 — Modèles, folds et comparaisons appariées
Un numéro de fold commun ne prouve pas un appariement. Le contrôle vérifie les trois partitions, les labels, les masques et la politique des seuils. Les autres paires sont signalées, sans calculer de différence.

In [ ]:
fold_metrics = a.metrics_by_fold(metrics, METRICS)
display(a.display_table(fold_metrics))
if not metrics.empty and METRICS[0] in metrics:
    display(a.display_table(metrics[metrics.status == "completed"].pivot_table(
        index=["task", "model", "noise_method", "configuration_name", "configuration_id", "protocol_id", "metric_source"],
        columns="fold", values=METRICS[0], aggfunc="mean")))
pairs = a.paired_comparisons(detail_runs, SPLIT, COHORT, METRICS[0])
display(a.display_table(pairs))
if comparison_runs:
    figures["fold_comparison/strip"] = a.plot_performance(comparison_metrics, METRICS[0], "strip")
    figures["fold_comparison/heatmap"] = a.plot_performance(comparison_metrics, METRICS[0], "heatmap")
    figures["fold_comparison/paired"] = a.plot_paired(pairs)
    plt.show()


## 6 — Analyses propres à chaque tâche
Binaire : métriques par classe et décision argmax sauvegardée. Multilabel : RR/RH séparés, macro/micro, exact match et Hamming, avec exclusion des annotations inconnues. Les scores continus restent accessibles lorsque les seuils sont inconnus.

In [ ]:
if not metrics.empty:
    for task, group in metrics.groupby(["task_type", "task"], dropna=False):
        print(task)
        columns = [c for c in group if c.startswith(("per_label.", "per_class.", "macro_", "micro_", "hamming_", "subset_"))]
        display(group[["run_name", "model", "fold", "metric_source"] + columns])


## 7 — Transfer learning
Les configurations sources, modules transférés et learning rates par phase sont conservés dans les colonnes structurées. `scratch` et `transfer` restent des stratégies de la même architecture.

In [ ]:
display(a.display_table(a.transfer_table(runs)))

## 8 — Prédictions, erreurs et divergences
Une ligne par amas et label. Les prédictions de validation interne ne sont jamais mélangées au test final ni considérées comme un ensemble OOF sans contrôle de couverture.

Les diagnostics binaires affichent uniquement la classe cible (DE, RR ou RH). Les matrices blanc–bleu indiquent le pourcentage **par classe réelle** (chaque ligne totalise 100 %), suivi du nombre d’amas entre parenthèses. Une classe sans amas porte « — (0) ». Les PNG individuels incluent le nom lisible du run, avec un suffixe technique pour éviter les collisions entre réexécutions.

La même figure inclut une rangée de courbes **loss**, **macro-F1** et **accuracy** par epoch, pour l’entraînement et la validation. Elle est également incluse dans le PNG exporté ; les métriques absentes sont signalées sans être reconstruites.

In [ ]:
predictions = a.prediction_table(runs, SPLIT, COHORT)
display(a.display_table(predictions))
internal_folds = [r for r in detail_runs if r.stage == "fold"]
if internal_folds:
    try:
        display(a.display_table(a.out_of_fold_predictions(internal_folds, COHORT)))
    except ValueError as error:
        print(f"Agrégation OOF indisponible : {error}")
for run in detail_runs:
    figures[f"confusion_matrices/{run.figure_stem}"] = a.plot_diagnostics(run, SPLIT, COHORT)
plt.show()
if len(detail_runs) > 1:
    try:
        display(a.compare_predictions(detail_runs, SPLIT, COHORT))
    except ValueError as error:
        print(f"Comparaison par amas non appariée : {error}")


### Analyses historiques approfondies (optionnelles)
Réutilise les audits d’annotations, les courbes de seuil sur validation, les combinaisons RR/RH et le bootstrap par amas existants. Les seuils test restent ceux des artefacts. Pour les runs binaires : diagnostic complet et comparaison de la contribution CHANDRA.

In [ ]:
LEGACY_ADVANCED = True
if LEGACY_ADVANCED:
    from clusterprep import multilabel_analysis as ma
    from clusterprep.analysis import binary
    binary_runs = []
    for run in detail_runs:
        if run.task_type == "multilabel" and run.thresholds is not None:
            try:
                legacy = ma.load_run(run.path)
                display(pd.DataFrame(ma.annotation_rows(legacy)))
                display(pd.DataFrame(ma.annotation_summary(legacy)))
                display(ma.threshold_curves(legacy))
                p = legacy.predictions[SPLIT].cohort(COHORT)
                display(ma.combination_scores(p, run.thresholds))
                display(ma.bootstrap_clusters(p, run.thresholds))
            except (ValueError, KeyError) as error:
                print(f"Analyse historique indisponible : {error}")
        elif run.task_type == "binary":
            legacy = binary.Run(run.path, run.metadata, run.metrics, run.splits, run.status)
            binary_runs.append(legacy)
            binary.detail(legacy, SPLIT, display_fn=display)
    if len(binary_runs) > 1:
        display(binary.gain_tables(binary_runs, split=SPLIT, cohort=COHORT))


## 9 — Galeries radio/CHANDRA
Activer `SHOW_GALLERIES`. Réutilisation du moteur existant pour images prétraitées alignées ; aucun chargement de checkpoint. Quatre galeries RR/RH ou TP/TN/FP/FN. `CHANDRA_ONLY` permet la vue X-ray seule ; une absence de CHANDRA conserve la radio. Les chemins et gabarits se configurent en section 1.

In [ ]:
if SHOW_GALLERIES:
    for run in detail_runs:
        gallery = a.plot_galleries(run, processed_root=PROCESSED_ROOT, split=SPLIT, cohort=COHORT,
            max_clusters=MAX_GALLERY_CLUSTERS, radio_template=RADIO_TEMPLATE,
            chandra_template=CHANDRA_TEMPLATE, chandra_only=CHANDRA_ONLY, view=GALLERY_VIEW)
        figures.update({f"galleries/{run.figure_stem}_{name}": fig for name, fig in gallery.items()})
    plt.show()


## 10 — Exports
Chaque exécution crée un sous-dossier unique hors des résultats sources. Les métadonnées, filtres et problèmes détectés permettent de retrouver les runs analysés.

La racine par défaut est `results/radio_noise`. `CLASSIFICATION_FILTER` distingue RR de RH ; `MODEL_FILTER`, `FOLD_FILTER` et `NOISE_FILTER` permettent de naviguer dans les expériences. Les comparaisons globales incluent tous les folds disponibles, indépendamment de la limite des diagnostics individuels.

In [ ]:
if EXPORT_TABLES or EXPORT_FIGURES:
    output = a.export_analysis(runs, EXPORT_ROOT, split=SPLIT, cohort=COHORT, metrics=METRICS,
        figures=figures if EXPORT_FIGURES else {}, export_tables=EXPORT_TABLES,
        settings={"root": str(ROOT_DIR), "recursive": RECURSIVE,
                  "detail_run_names": [r.run_name for r in detail_runs],
                  "detail_run_ids": [r.run_id for r in detail_runs]})
    issues.to_csv(output / "issues.csv", index=False)
    pairs.to_csv(output / "paired_comparisons.csv", index=False)
    print(output)
plt.close("all")
